# Baseline de elegibilidad de crédito
Hackathon Factored AI & Data 2026. Tema elegido por el equipo: **crédito** (oferta preaprobada, recálculo en vivo de capacidad de pago y derivación a un humano).

**Qué es y qué no es este baseline**
- La política es **sintética y provisional** (`policy/credit_policy.yaml`), definida por el equipo. No hay resultados reales de solicitudes de crédito en los datos, así que **no existe una verdad de terreno externa**.
- Por eso el baseline mide **el comportamiento de la política sobre la base de clientes** y lo compara con dos referencias simples. La evaluación del chat consistirá en verificar que reproduce fielmente lo que decide el motor (sin inventar reglas ni aprobar por su cuenta).
- No se entrena ningún modelo de riesgo: en estos datos `credit_score` no correlaciona con `days_past_due` (r = -0,004).
- Medición offline sobre datos sintéticos. No es una mejora medida en producción.

In [1]:
import sys, json, warnings
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
HERE = Path('.').resolve()                      # analysis/notebooks (el notebook se ejecuta aqui)
sys.path.insert(0, str(HERE.parent)); sys.path.insert(0, str(HERE.parents[1] / 'backend'))
from paths import RAW as DATA, WORK as GOLD     # ver analysis/paths.py (HACKATHON_RAW_DIR / HACKATHON_WORK_DIR)
from app.policy import credit_engine as ce
OUT = HERE / 'outputs'; OUT.mkdir(exist_ok=True)
POLICY = ce.load_policy()
print('política', POLICY['version'], '| sintética:', POLICY['synthetic'])

política 0.1.0-provisional | sintética: True


## 1. Carga y contrato de datos

In [2]:
cu = pd.read_csv(DATA/'customers.csv', encoding='utf-8-sig')
pr = pd.read_csv(DATA/'products.csv', encoding='utf-8-sig')
fx = pd.read_csv(DATA/'daily_exchange_rates.csv', encoding='utf-8-sig')
cu['country'] = cu.country.replace({'Mexico': 'México'})

checks = {
 'customers: PK única': cu.customer_id.is_unique,
 'products: PK única': pr.product_id.is_unique,
 'products: customer_id existe en customers': pr.customer_id.isin(cu.customer_id).all(),
 'customers: credit_score en 300-850 (no nulos)': cu.credit_score.dropna().between(300, 850).all(),
 'customers: ingreso >= 0 (no nulos)': (cu.estimated_monthly_income.dropna() >= 0).all(),
 'products: days_past_due >= 0 (no nulos)': (pr.days_past_due.dropna() >= 0).all(),
 'products: current_balance >= 0': (pr.current_balance.dropna() >= 0).all(),
 'fx: existe tasa para cada par de monedas en la última fecha': fx[fx.date == fx.date.max()].shape[0] == 12,
}
qc = pd.Series(checks, name='ok').map({True:'PASS', False:'FAIL'}).to_frame(); display(qc)
print('Nulos: credit_score %.1f%% | ingreso %.1f%%' % (100*cu.credit_score.isna().mean(), 100*cu.estimated_monthly_income.isna().mean()))
print('Clientes sin productos:', (~cu.customer_id.isin(pr.customer_id)).sum())
m = pr.merge(cu[['customer_id','country']], on='customer_id')
print('Moneda de productos por país del cliente (hallazgo: todos los productos de México están en USD):')
display(pd.crosstab(m.country, m.currency))

,ok
customers: PK única,PASS
products: PK única,PASS
products: customer_id existe en customers,PASS
customers: credit_score en 300-850 (no nulos),PASS
customers: ingreso >= 0 (no nulos),PASS
products: days_past_due >= 0 (no nulos),PASS
products: current_balance >= 0,PASS
fx: existe tasa para cada par de monedas en la última fecha,PASS


Nulos: credit_score 15.0% | ingreso 20.0%


Clientes sin productos: 10422
Moneda de productos por país del cliente (hallazgo: todos los productos de México están en USD):


currency,ARS,COP,USD
country,,,
Argentina,71524,0,7918
Colombia,0,107975,12185
México,0,0,200398


## 2. Capa gold: perfil crediticio por cliente
Se construye una tabla por cliente con lo que necesita el motor. Es la tabla que el chat consultará (no las transaccionales).

**Supuestos declarados** (el dataset no trae cuotas): cuota mínima de tarjeta = 5% del saldo; préstamos personales a 36 meses restantes; hipotecas a 180 meses; tasa del propio producto. Los saldos en USD o en otra moneda se convierten a la moneda del ingreso con la tasa de la **última fecha** disponible.

In [3]:
TYPE_MAP = {'Tarjeta Crédito': 'credit_card', 'Préstamo Personal': 'personal_loan', 'Préstamo Hipotecario': 'mortgage'}
INCOME_CCY = {'México': 'MXN', 'Colombia': 'COP', 'Argentina': 'ARS'}
last = fx[fx.date == fx.date.max()]
rate = {(r.source_currency, r.target_currency): r.exchange_rate for r in last.itertuples()}
def convert(amount, src, dst):
    return amount if src == dst else amount * rate[(src, dst)]

A = POLICY['existing_debt_assumptions']
pr['ptype'] = pr.product_type.map(TYPE_MAP)
cr = pr[pr.ptype.notna() & (pr.product_status == 'Active')].merge(cu[['customer_id','country']], on='customer_id')
cr['income_ccy'] = cr.country.map(INCOME_CCY)
cr['balance_inc'] = [convert(b, s, d) for b, s, d in zip(cr.current_balance.fillna(0), cr.currency, cr.income_ccy)]
def monthly_debt(r):
    if r.ptype == 'credit_card': return r.balance_inc * A['card_min_payment_pct']
    months = A['personal_loan_remaining_months'] if r.ptype == 'personal_loan' else A['mortgage_remaining_months']
    return ce.pmt(r.balance_inc, r.interest_rate if r.interest_rate == r.interest_rate else 0.0, months)
cr['monthly_debt'] = cr.apply(monthly_debt, axis=1)
debt = cr.groupby('customer_id').monthly_debt.sum().rename('existing_monthly_debt')

# mora: máximo de días entre todos los productos de crédito del cliente (no cerrados)
credit_all = pr[pr.ptype.notna() & (pr.product_status != 'Closed')]
dpd = credit_all.groupby('customer_id').days_past_due.max().rename('max_days_past_due')
nact = pr[pr.product_status == 'Active'].groupby('customer_id').size().rename('n_active_products')

prof = cu[['customer_id','country','segment','customer_status','credit_score','estimated_monthly_income','gender','date_of_birth']].copy()
prof = prof.join(debt, on='customer_id').join(dpd, on='customer_id').join(nact, on='customer_id')
prof['existing_monthly_debt'] = prof.existing_monthly_debt.fillna(0.0)
prof['max_days_past_due'] = prof.max_days_past_due.fillna(0)
prof['n_active_products'] = prof.n_active_products.fillna(0).astype(int)
prof = prof.rename(columns={'estimated_monthly_income': 'monthly_income'})
prof['income_ccy'] = prof.country.map(INCOME_CCY)
prof['dti_existing'] = prof.existing_monthly_debt / prof.monthly_income
prof['score_band'] = prof.credit_score.map(lambda s: ce.score_band(s, POLICY) if s == s else np.nan)
prof.to_parquet(GOLD/'gold_customer_credit_profile.parquet')
print('gold_customer_credit_profile:', prof.shape)
display(prof[['monthly_income','existing_monthly_debt','dti_existing','max_days_past_due','n_active_products']].describe().round(3))

gold_customer_credit_profile: (150000, 14)


,monthly_income,existing_monthly_debt,dti_existing,max_days_past_due,n_active_products
count,1.199670e+05,1.500000e+05,119967.000,150000.000,150000.000
mean,5.895962e+06,1.637766e+05,0.065,9.170,2.266
std,1.447195e+07,6.180712e+05,0.184,32.103,1.506
min,5.100280e+03,0.000000e+00,0.000,0.000,0.000
25%,3.922756e+04,0.000000e+00,0.000,0.000,1.000
50%,3.429744e+05,6.303510e+02,0.004,0.000,2.000
75%,5.362112e+06,2.000381e+04,0.054,0.000,3.000
max,1.118951e+08,1.256661e+07,9.089,180.000,11.000


In [4]:
# Sanidad de la estimación de deuda: ¿qué tan realista queda la carga existente?
ok = prof.dti_existing.replace([np.inf, -np.inf], np.nan).dropna()
print('DTI existente (deuda/ingreso): p50=%.3f p75=%.3f p95=%.3f | >20%%: %.1f%% | >100%%: %.1f%%' % (
    ok.median(), ok.quantile(.75), ok.quantile(.95), 100*(ok > .20).mean(), 100*(ok > 1).mean()))
print('Bandas de score (%):', (prof.score_band.value_counts(normalize=True, dropna=False).sort_index()*100).round(1).to_dict())

DTI existente (deuda/ingreso): p50=0.004 p75=0.054 p95=0.317 | >20%: 8.2% | >100%: 0.7%
Bandas de score (%): {1.0: 1.2, 2.0: 24.8, 3.0: 33.7, 4.0: 16.7, 5.0: 8.7, nan: 15.0}


## 3. Resultado de la política sobre la solicitud tipo
Solicitud tipo: préstamo personal a 36 meses por 3× el ingreso mensual (parámetro en la política). Ingreso **en archivo** (no declarado).

In [5]:
REQ0 = POLICY['baseline_request']
def run(df, income_col='monthly_income', declared=False, amount_mult=None):
    mult = amount_mult or REQ0['amount_income_multiple']
    out = []
    for r in df.itertuples():
        inc = getattr(r, income_col)
        app = dict(customer_status=r.customer_status, credit_score=r.credit_score, monthly_income=inc,
                   income_is_declared=declared, existing_monthly_debt=r.existing_monthly_debt,
                   max_days_past_due=r.max_days_past_due, n_active_products=r.n_active_products)
        req = dict(product=REQ0['product'], months=REQ0['months'], amount=(inc*mult) if inc == inc else 0.0)
        d = ce.evaluate(app, req, POLICY)
        out.append((d.outcome, d.reasons[0] if d.reasons else '', d.band, d.rate_pct, d.dti_after, d.max_amount))
    return pd.DataFrame(out, columns=['outcome','reason','band','rate_pct','dti_after','max_amount'], index=df.index)
res0 = run(prof)
prof = prof.join(res0)
tab = (prof.outcome.value_counts(normalize=True)*100).round(2)
print('Resultado por cliente (%):'); display(tab.to_frame('%'))
display((prof.reason.value_counts(normalize=True)*100).round(2).to_frame('% por motivo principal'))

Resultado por cliente (%):


,%
outcome,
eligible,44.24
needs_data,27.20
needs_review,18.46
declined,10.10


,% por motivo principal
reason,
WITHIN_LIMIT,44.24
MISSING_DATA,27.20
CUSTOMER_NOT_ACTIVE,14.87
DTI_EXCEEDED,7.21
DELINQUENT_REVIEW,2.18
DELINQUENT_OVER_LIMIT,2.16
BORDERLINE_DTI,1.41
LOW_SCORE_BAND,0.73


In [6]:
by_country = pd.crosstab(prof.country, prof.outcome, normalize='index').mul(100).round(1)
by_segment = pd.crosstab(prof.segment, prof.outcome, normalize='index').mul(100).round(1)
display(by_country); display(by_segment)
el = prof[prof.outcome.isin(['eligible','eligible_provisional'])]
print('Tasa asignada a elegibles (%%): p10=%.1f p50=%.1f p90=%.1f' % tuple(el.rate_pct.quantile([.1,.5,.9])))

outcome,declined,eligible,needs_data,needs_review
country,,,,
Argentina,10.0,44.1,27.2,18.6
Colombia,10.0,44.2,27.3,18.5
México,10.2,44.3,27.1,18.3


outcome,declined,eligible,needs_data,needs_review
segment,,,,
Basic,12.8,41.3,27.1,18.8
Plus,4.5,50.0,27.5,18.0
Premium,2.3,53.2,27.2,17.3
Student,21.7,32.5,27.1,18.7


Tasa asignada a elegibles (%): p10=18.0 p50=23.0 p90=26.0


## 4. Referencias (baselines) del sistema
- **B0 – todo a humano:** el 100% de las consultas de crédito llegan a un agente. Es la línea de base operativa actual conocida (sin automatización).
- **B1 – preaprobación estática por segmento:** Premium y Plus preaprobados, Basic y Student no. No mira ingreso, deuda ni mora. Sirve para mostrar el riesgo de una regla sin cálculo.
- **Sistema propuesto:** motor de política con datos del cliente + recálculo en el chat.

In [7]:
prof['b1_preapproved'] = prof.segment.isin(['Premium','Plus'])
prof['engine_ok'] = prof.outcome.isin(['eligible','eligible_provisional'])
cmp = pd.crosstab(prof.b1_preapproved.map({True:'B1 preaprueba', False:'B1 no preaprueba'}),
                  prof.outcome, margins=True)
display(cmp)
pre = prof[prof.b1_preapproved]
print('De los preaprobados por B1: el motor los rechaza %.1f%%, pide datos %.1f%%, revisión humana %.1f%%' % (
    100*(pre.outcome=='declined').mean(), 100*(pre.outcome=='needs_data').mean(), 100*(pre.outcome=='needs_review').mean()))
nopre = prof[~prof.b1_preapproved]
print('De los NO preaprobados por B1: el motor los considera elegibles %.1f%% (oportunidad perdida)' % (100*nopre.engine_ok.mean()))
handoff = {'B0': 1.0, 'motor': float(prof.outcome.isin(['needs_review']).mean()),
           'motor_incl_needs_data': float(prof.outcome.isin(['needs_review','needs_data']).mean())}
print('Derivación a humano -> B0: 100%% | motor (solo revisión): %.1f%% | motor + datos faltantes: %.1f%%' % (
    100*handoff['motor'], 100*handoff['motor_incl_needs_data']))

outcome,declined,eligible,needs_data,needs_review,All
b1_preapproved,,,,,
B1 no preaprueba,13091,39513,26352,18290,97246
B1 preaprueba,2060,26850,14451,9393,52754
All,15151,66363,40803,27683,150000


De los preaprobados por B1: el motor los rechaza 3.9%, pide datos 27.4%, revisión humana 17.8%
De los NO preaprobados por B1: el motor los considera elegibles 40.6% (oportunidad perdida)
Derivación a humano -> B0: 100% | motor (solo revisión): 18.5% | motor + datos faltantes: 45.7%


## 5. Recálculo en vivo: sensibilidad a datos que aporta el cliente
Caso de uso de la reunión: el cliente declara más ingreso (aumento de sueldo, ingreso de la pareja). El motor recalcula con ingreso **declarado (no verificado)**, por lo que un resultado favorable es *provisional*. Se cuenta cuántos rechazos por capacidad de pago cambian.

In [8]:
dti_rej = prof[(prof.outcome == 'declined') & (prof.reason == 'DTI_EXCEEDED')].copy()
print('Rechazados por capacidad de pago:', len(dti_rej))
rows = []
for uplift in [0.10, 0.25, 0.50, 1.00]:
    dti_rej['inc_decl'] = dti_rej.monthly_income * (1 + uplift)
    r = run(dti_rej, income_col='inc_decl', declared=True, amount_mult=REQ0['amount_income_multiple']*1/(1+uplift))
    # se mantiene el MONTO solicitado original: amount = 3 * ingreso original
    rows.append(dict(aumento_declarado=f'+{int(uplift*100)}%',
                     pasan_a_provisional=(r.outcome=='eligible_provisional').mean(),
                     pasan_a_revision=(r.outcome=='needs_review').mean(),
                     siguen_rechazados=(r.outcome=='declined').mean()))
sens = pd.DataFrame(rows).set_index('aumento_declarado'); display((sens*100).round(1))
print('Regla: aumentos declarados mayores a', int(POLICY['declared_income']['max_uplift_without_review']*100),
      '% no deberían resolverse solos; el orquestador los envía a revisión humana.')

Rechazados por capacidad de pago: 10810


,pasan_a_provisional,pasan_a_revision,siguen_rechazados
aumento_declarado,,,
+10%,0.0,15.6,84.4
+25%,20.2,11.4,68.3
+50%,40.7,8.8,50.6
+100%,63.0,5.1,31.9


Regla: aumentos declarados mayores a 50 % no deberían resolverse solos; el orquestador los envía a revisión humana.


## 6. Auditoría de equidad (solo medición)
Los atributos protegidos **no entran** a la política. Se mide si los resultados difieren por grupo para detectar disparidades de fondo (por ejemplo, vía ingreso).

In [9]:
prof['age'] = (pd.Timestamp('2026-06-17') - pd.to_datetime(prof.date_of_birth)).dt.days // 365
prof['age_band'] = pd.cut(prof.age, [17, 25, 35, 45, 55, 65, 120], labels=['18-25','26-35','36-45','46-55','56-65','66+'])
def rate_tab(col):
    g = prof.groupby(col, observed=True)
    return pd.DataFrame({'n': g.size(), 'elegibles_%': g.engine_ok.mean()*100,
                         'rechazados_%': g.apply(lambda x:(x.outcome=='declined').mean()*100),
                         'datos_faltantes_%': g.apply(lambda x:(x.outcome=='needs_data').mean()*100)}).round(1)
for c in ['gender','country','age_band']: print(c); display(rate_tab(c))

gender


,n,elegibles_%,rechazados_%,datos_faltantes_%
gender,,,,
F,50508,44.0,10.3,27.1
M,49684,44.6,9.9,27.2
O,49808,44.1,10.1,27.3


country


,n,elegibles_%,rechazados_%,datos_faltantes_%
country,,,,
Argentina,29842,44.1,10.0,27.2
Colombia,45251,44.2,10.0,27.3
México,74907,44.3,10.2,27.1


age_band


,n,elegibles_%,rechazados_%,datos_faltantes_%
age_band,,,,
18-25,11892,44.7,10.2,27.0
26-35,23676,44.3,10.2,27.2
36-45,23736,44.1,10.0,27.1
46-55,23920,44.3,10.2,27.5
56-65,23886,44.4,10.2,26.9
66+,42890,44.1,10.0,27.3


## 7. Resumen guardado

In [10]:
summary = {
  'politica': {'version': POLICY['version'], 'sintetica': True, 'max_dti': POLICY['max_dti']},
  'solicitud_tipo': REQ0,
  'clientes': int(len(prof)),
  'resultado_pct': tab.to_dict(),
  'derivacion_humano': handoff,
  'b1_preaprobados_rechazados_por_motor_pct': float(100*(pre.outcome=='declined').mean()),
  'sensibilidad_ingreso_declarado_pct': (sens*100).round(1).to_dict(),
  'supuestos_deuda': A,
  'limites': [
    'Politica sintetica y provisional; no hay verdad de terreno externa.',
    'El dataset no trae cuotas: la deuda existente se estima con supuestos declarados.',
    'Todos los productos de Mexico estan en USD; se convierten con la tasa de la ultima fecha.',
    'credit_score no correlaciona con days_past_due: no se entrena modelo de riesgo.',
    'Medicion offline sobre datos sinteticos, no mejora en produccion.',
  ]}
(OUT/'credit_baseline_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=float), encoding='utf-8')
print('guardado', OUT/'credit_baseline_summary.json')

guardado C:\dev\hackathon\repo\analysis\notebooks\outputs\credit_baseline_summary.json
